# Versioned document-level legal document parsing

This Colab notebook is the research configuration and execution driver. The implementation lives in the Python modules beside this notebook under `src/document_split`:

- `config.py` — settings and Arrow schemas
- `processing.py` — RTF conversion, complete-document model validation, and one-row Parquet construction
- `cloud.py` — immutable manifests, BigQuery streaming, GCS I/O, and run logs
- `runtime.py` — Colab authentication and model loading
- `pipeline.py` — end-to-end orchestration
- `self_checks.py` — local non-GPU verification

In [ ]:
# Run once after selecting a GPU runtime in Colab.
# This parser is text-only. Remove Colab's optional TorchAudio package so a
# mismatched CUDA build cannot prevent Transformers from importing.
%pip uninstall -q -y torchaudio
%pip install -q "transformers>=4.50.0" accelerate huggingface_hub google-cloud-bigquery google-cloud-storage striprtf pyarrow tqdm json-repair

## Import the local modules

The bootstrap supports starting Jupyter from the repository root, `src`, or `src/document_split`. In Colab, clone or mount the repository before running this cell.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
module_roots = [cwd / "src", cwd, cwd.parent]
for module_root in module_roots:
    if (module_root / "document_split" / "__init__.py").exists():
        if str(module_root) not in sys.path:
            sys.path.insert(0, str(module_root))
        break
else:
    raise RuntimeError(
        "Could not find src/document_split. Clone or mount the repository, "
        "then run the notebook from the repository root or src/document_split."
    )

from document_split import (
    CRIMINAL_PROMPT,
    CRIMINAL_SCHEMA,
    ExtractionSettings,
    StorageSettings,
    run_pipeline,
    run_self_checks,
)

## Research prompt, schema, and version

The production criminal-law prompt and its canonical PyArrow schema are imported together. Adding, removing, or retyping a field requires a new `info_version_N`.

In [ ]:
EXTRACTION_PROMPT = CRIMINAL_PROMPT

# Use the canonical criminal-law schema defined alongside the prompt.
EXTRACTION_SCHEMA = CRIMINAL_SCHEMA

EXTRACTION_SETTINGS = ExtractionSettings(
    prompt=EXTRACTION_PROMPT,
    extraction_schema=EXTRACTION_SCHEMA,
    max_new_tokens=32_768,
    json_retries=0,
)

STORAGE_SETTINGS = StorageSettings(
    info_version="info_version_8",
    justice_kinds=(2,),
    limit=None,
    skip_existing=True,
)

EXTRACTION_SETTINGS.output_schema

## Non-GPU self-checks

These tests do not access BigQuery, GCS, Colab secrets, or the model.

In [ ]:
run_self_checks()

## Start a production run

1. Replace `EXTRACTION_PROMPT` and rerun the configuration cell.
2. Confirm the schema and increment `info_version_N` if the research definition changed.
3. Add a Colab secret named `cloud_access` containing the service-account JSON. Optionally add `HF_TOKEN`.
4. Use a small `limit` smoke test before starting the full resumable run.

In [ ]:
# Safe smoke test (uncomment after configuring the prompt and secrets):
# smoke_storage = StorageSettings(
#     info_version=STORAGE_SETTINGS.info_version,
#     justice_kinds=STORAGE_SETTINGS.justice_kinds,
#     limit=5,
# )
# smoke_result = run_pipeline(EXTRACTION_SETTINGS, smoke_storage)
# print(smoke_result)

# Full resumable run:
# result = run_pipeline(EXTRACTION_SETTINGS, STORAGE_SETTINGS)
# print(result)